In [1]:
# | default_exp gsc.queries

# GSC Queries
> Analytics query functions for Google Search Console data.

In [2]:
#| export
from sqlmodel import Session, select
from seootter.models import GSCAnalytics
from datetime import datetime, timedelta
from functools import partial
from fastcore.basics import compose
from sqlalchemy import func
from seootter.gsc.filters import filter_site, filter_dates, filter_dimension, filter_exclude_pages
from seootter.gsc.filters import AnalyticsSummary

In [3]:
# | export
def get_top_queries(session: Session,  # Active database session
                    site_url: str,  # GSC property URL
                    start_date: str,  # Start date (YYYY-MM-DD)
                    end_date: str,  # End date (YYYY-MM-DD)
                    country: str | None = None,  # Filter by country code
                    page_path: str | None = None,  # Filter by page path substring
                    limit: int = 10,  # Max rows to return
                    sort_by: str = "clicks"  # Sort by 'clicks' or 'impressions'
                    ) -> list[dict]:
    "Get top performing queries, optionally filtered by country and page."
    base = select(GSCAnalytics.query,
                  func.sum(GSCAnalytics.clicks).label("total_clicks"),
                  func.sum(GSCAnalytics.impressions).label("total_impressions"),
                  func.avg(GSCAnalytics.position).label("avg_position"),
                  func.avg(GSCAnalytics.ctr).label("avg_ctr"))
    filters = [partial(filter_site, site_url=site_url),
               partial(filter_dates, start=start_date, end=end_date)]
    if country: filters.append(partial(filter_dimension, dimension="country", value=country))
    if page_path: filters.append(lambda q: q.where(GSCAnalytics.page.contains(page_path)))
    sort_col = func.sum(GSCAnalytics.impressions) if sort_by == "impressions" else func.sum(GSCAnalytics.clicks)
    query = (compose(*filters)(base)
             .where(GSCAnalytics.query.isnot(None))
             .group_by(GSCAnalytics.query)
             .order_by(sort_col.desc())
             .limit(limit))
    return [row._asdict() for row in session.exec(query)]

In [4]:
#| export
def get_top_pages(session: Session, site_url: str, start_date: str, end_date: str,
                  country: str | None = None, limit: int = 20, sort_by: str = "clicks") -> list[dict]:
    "Get top performing pages by clicks or impressions."
    base = select(GSCAnalytics.page,
                  func.sum(GSCAnalytics.clicks).label("total_clicks"),
                  func.sum(GSCAnalytics.impressions).label("total_impressions"),
                  func.avg(GSCAnalytics.position).label("avg_position"),
                  func.avg(GSCAnalytics.ctr).label("avg_ctr"))
    filters = [partial(filter_site, site_url=site_url),
               partial(filter_dates, start=start_date, end=end_date)]
    if country: filters.append(partial(filter_dimension, dimension="country", value=country))
    sort_col = func.sum(GSCAnalytics.impressions) if sort_by == "impressions" else func.sum(GSCAnalytics.clicks)
    query = (compose(*filters)(base)
             .where(GSCAnalytics.page.isnot(None))
             .group_by(GSCAnalytics.page)
             .order_by(sort_col.desc())
             .limit(limit))
    return [row._asdict() for row in session.exec(query)]

In [5]:
#| export
def get_wins(session: Session, site_url: str, start_date: str, end_date: str,
             min_impressions: int = 100, min_position: float = 10.0,
             max_position: float = 50.0, country: str | None = None,
             page_url: str | None = None, limit: int = 20) -> list[dict]:
    "Get high-impression, low-ranking keyword opportunities."
    base = select(GSCAnalytics.query,
                  func.sum(GSCAnalytics.clicks).label("total_clicks"),
                  func.sum(GSCAnalytics.impressions).label("total_impressions"),
                  func.avg(GSCAnalytics.position).label("avg_position"),
                  func.avg(GSCAnalytics.ctr).label("avg_ctr"))
    filters = [partial(filter_site, site_url=site_url),
               partial(filter_dates, start=start_date, end=end_date)]
    if country: filters.append(partial(filter_dimension, dimension="country", value=country))
    if page_url: filters.append(lambda q: q.where(GSCAnalytics.page == page_url))
    query = (compose(*filters)(base)
             .where(GSCAnalytics.query.isnot(None))
             .group_by(GSCAnalytics.query)
             .having(func.sum(GSCAnalytics.impressions) >= min_impressions,
                     func.avg(GSCAnalytics.position) >= min_position,
                     func.avg(GSCAnalytics.position) <= max_position)
             .order_by(func.sum(GSCAnalytics.impressions).desc())
             .limit(limit))
    return [row._asdict() for row in session.exec(query)]

In [6]:
# | export
def get_top_queries_excluding_pages(session: Session,  # Active database session
                                    site_url: str,  # GSC property URL
                                    start_date: str,  # Start date (YYYY-MM-DD)
                                    end_date: str,  # End date (YYYY-MM-DD)
                                    exclude_pages: list[str],  # Page substrings to exclude
                                    country: str | None = None,  # Filter by country code
                                    limit: int = 10  # Max rows to return
                                    ) -> list[dict]:
    "Get top queries excluding specific pages."
    base = select(GSCAnalytics.query,
                  func.sum(GSCAnalytics.clicks).label("total_clicks"),
                  func.sum(GSCAnalytics.impressions).label("total_impressions"))
    filters = [partial(filter_site, site_url=site_url),
               partial(filter_dates, start=start_date, end=end_date),
               partial(filter_exclude_pages, exclude_pages=exclude_pages)]
    if country: filters.append(partial(filter_dimension, dimension="country", value=country))
    query = (compose(*filters)(base)
             .where(GSCAnalytics.query.isnot(None))
             .group_by(GSCAnalytics.query)
             .order_by(func.sum(GSCAnalytics.clicks).desc())
             .limit(limit))
    return [row._asdict() for row in session.exec(query)]

In [7]:
# | export
def get_page_analytics(session: Session,  # Active database session
                       site_url: str,  # GSC property URL
                       page_path: str,  # Partial page path to match
                       start_date: str,  # Start date (YYYY-MM-DD)
                       end_date: str  # End date (YYYY-MM-DD)
                       ) -> dict:
    "Get aggregated analytics and top queries for a specific page."
    filters = compose(
        partial(filter_site, site_url=site_url),
        partial(filter_dates, start=start_date, end=end_date),
    )
    base = filters(select(
        func.sum(GSCAnalytics.clicks).label("total_clicks"),
        func.sum(GSCAnalytics.impressions).label("total_impressions"),
        func.avg(GSCAnalytics.position).label("avg_position"),
        func.avg(GSCAnalytics.ctr).label("avg_ctr"),
    )).where(GSCAnalytics.page.contains(page_path))
    row = session.exec(base).first()

    top_queries = session.exec(
        filters(select(GSCAnalytics.query, func.sum(GSCAnalytics.clicks).label("clicks"))
                .where(GSCAnalytics.page.contains(page_path), GSCAnalytics.query.isnot(None))
                .group_by(GSCAnalytics.query)
                .order_by(func.sum(GSCAnalytics.clicks).desc())
                .limit(10))
    ).all()

    return {"page_path": page_path,
            "total_clicks": row.total_clicks or 0,
            "total_impressions": row.total_impressions or 0,
            "avg_position": row.avg_position or 0,
            "avg_ctr": row.avg_ctr or 0,
            "top_queries": [q for q, _ in top_queries]}

In [8]:
# | export
def get_analytics_by_date_range(session: Session,  # Active database session
                                site_url: str,  # GSC property URL
                                start_date: str,  # Start date (YYYY-MM-DD)
                                end_date: str  # End date (YYYY-MM-DD)
                                ) -> list[GSCAnalytics]:
    "Get all raw GSC analytics rows for a date range."
    return session.exec(
        compose(partial(filter_site, site_url=site_url),
                partial(filter_dates, start=start_date, end=end_date)
                )(select(GSCAnalytics))
    ).all()

In [9]:
# | export
def get_trends(session: Session,  # Active database session
               site_url: str,  # GSC property URL
               start_date: str,  # Start date (YYYY-MM-DD)
               end_date: str,  # End date (YYYY-MM-DD)
               dimension: str | None = None  # Optional dimension to group by
               ) -> list[dict]:
    "Get click/impression trends over time, optionally grouped by a dimension."
    columns = [GSCAnalytics.date,
               func.sum(GSCAnalytics.clicks).label("clicks"),
               func.sum(GSCAnalytics.impressions).label("impressions"),
               func.avg(GSCAnalytics.position).label("avg_position"),
               func.avg(GSCAnalytics.ctr).label("avg_ctr")]
    group_by = [GSCAnalytics.date]
    if dimension:
        dim_col = getattr(GSCAnalytics, dimension)
        columns.insert(1, dim_col)
        group_by.append(dim_col)
    query = (compose(partial(filter_site, site_url=site_url),
                     partial(filter_dates, start=start_date, end=end_date))
             (select(*columns))
             .group_by(*group_by)
             .order_by(GSCAnalytics.date))
    return [row._asdict() for row in session.exec(query)]

In [10]:
# | export
def get_analytics_by(session: Session,  # Active database session
                     site_url: str,  # GSC property URL
                     start_date: str,  # Start date (YYYY-MM-DD)
                     end_date: str,  # End date (YYYY-MM-DD)
                     dimension: str,  # GSCAnalytics field to filter by
                     value: str  # Value to match
                     ) -> list[AnalyticsSummary]:
    "Get query-level analytics filtered by a specific dimension value."
    base = select(GSCAnalytics.query,
                  func.sum(GSCAnalytics.clicks).label("clicks"),
                  func.sum(GSCAnalytics.impressions).label("impressions"))
    query = (compose(partial(filter_site, site_url=site_url),
                     partial(filter_dates, start=start_date, end=end_date),
                     partial(filter_dimension, dimension=dimension, value=value))
             (base).group_by(GSCAnalytics.query))
    return [AnalyticsSummary(**row._asdict()) for row in session.exec(query)]

In [11]:
#| export
def compare_date_ranges(session: Session,  # Active database session
                        site_url: str,  # GSC property URL
                        start1: str,  # First period start date (YYYY-MM-DD)
                        end1: str,  # First period end date (YYYY-MM-DD)
                        start2: str,  # Second period start date (YYYY-MM-DD)
                        end2: str,  # Second period end date (YYYY-MM-DD)
                        page_url: str | None = None  # Optional specific page to compare
                        ) -> dict:
    "Compare GSC metrics between two date ranges, optionally for a specific page."
    base = select(
        func.sum(GSCAnalytics.clicks).label("clicks"),
        func.sum(GSCAnalytics.impressions).label("impressions"),
        func.avg(GSCAnalytics.position).label("avg_position"),
        func.avg(GSCAnalytics.ctr).label("avg_ctr")
    )
    filters1 = compose(
        partial(filter_site, site_url=site_url),
        partial(filter_dates, start=start1, end=end1)
    )
    if page_url:
        filters1 = compose(filters1, lambda q: q.where(GSCAnalytics.page == page_url))
    row1 = session.exec(filters1(base)).first()

    filters2 = compose(
        partial(filter_site, site_url=site_url),
        partial(filter_dates, start=start2, end=end2)
    )
    if page_url:
        filters2 = compose(filters2, lambda q: q.where(GSCAnalytics.page == page_url))
    row2 = session.exec(filters2(base)).first()

    def to_dict(row):
        return {
            "clicks": row.clicks or 0,
            "impressions": row.impressions or 0,
            "avg_position": round(row.avg_position or 0, 2),
            "avg_ctr": round((row.avg_ctr or 0) * 100, 2)
        }

    return {
        "page_url": page_url,
        "period1": {"start": start1, "end": end1, **to_dict(row1)},
        "period2": {"start": start2, "end": end2, **to_dict(row2)}
    }

In [12]:
#| export
def get_country_breakdown(session: Session,  # Active database session
                          site_url: str,  # GSC property URL
                          start_date: str,  # Start date (YYYY-MM-DD)
                          end_date: str,  # End date (YYYY-MM-DD)
                          page_url: str | None = None,  # Optional specific page to filter
                          limit: int = 20  # Max number of countries to return
                          ) -> list[dict]:
    "Get traffic metrics grouped by country, optionally for a specific page."
    base = select(
        GSCAnalytics.country,
        func.sum(GSCAnalytics.clicks).label("clicks"),
        func.sum(GSCAnalytics.impressions).label("impressions"),
        func.avg(GSCAnalytics.position).label("avg_position"),
        func.avg(GSCAnalytics.ctr).label("avg_ctr")
    )
    filters = compose(
        partial(filter_site, site_url=site_url),
        partial(filter_dates, start=start_date, end=end_date)
    )
    if page_url:
        filters = compose(filters, lambda q: q.where(GSCAnalytics.page == page_url))

    query = (filters(base)
             .where(GSCAnalytics.country.isnot(None))
             .group_by(GSCAnalytics.country)
             .order_by(func.sum(GSCAnalytics.clicks).desc())
             .limit(limit))

    return [row._asdict() for row in session.exec(query)]